In [2]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

usda_path = DATA_RAW / "StateAndCountyData.csv"

usda = pd.read_csv(usda_path)

print("USDA shape:", usda.shape)
display(usda.head())

USDA shape: (957753, 5)


,FIPS,State,County,Variable_Code,Value
0,1001.0,AL,Autauga,LACCESS_POP15,18092.66135
1,1001.0,AL,Autauga,LACCESS_POP19,18503.22551
2,1001.0,AL,Autauga,PCH_LACCESS_POP_15_19,2.26923
3,1001.0,AL,Autauga,PCT_LACCESS_POP15,33.15435
4,1001.0,AL,Autauga,PCT_LACCESS_POP19,33.90670


In [3]:
usda.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 957753 entries, 0 to 957752
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   FIPS           957278 non-null  float64
 1   State          957753 non-null  object 
 2   County         957753 non-null  object 
 3   Variable_Code  957753 non-null  object 
 4   Value          957753 non-null  float64
dtypes: float64(2), object(3)
memory usage: 36.5+ MB


In [4]:
print("Unique FIPS:", usda["FIPS"].nunique())
print("Unique states:", usda["State"].nunique())
print("Unique variable codes:", usda["Variable_Code"].nunique())

Unique FIPS: 3156
Unique states: 51
Unique variable codes: 304


In [5]:
missing_fips = usda[usda["FIPS"].isna()]

print("Rows with missing FIPS:", len(missing_fips))
display(missing_fips.head(20))

Rows with missing FIPS: 475


,FIPS,State,County,Variable_Code,Value
430834,NaN,AK,Aleutian Islands Area,DIRSALES_FARMS12,-9999.0
430835,NaN,AK,Aleutian Islands Area,DIRSALES_FARMS17,-9999.0
430836,NaN,AK,Aleutian Islands Area,PCH_DIRSALES_FARMS_12_17,-9999.0
430837,NaN,AK,Aleutian Islands Area,PCT_LOCLFARM12,-9999.0
430838,NaN,AK,Aleutian Islands Area,PCT_LOCLFARM17,-9999.0
430839,NaN,AK,Aleutian Islands Area,PCT_LOCLSALE12,-9999.0
430840,NaN,AK,Aleutian Islands Area,PCT_LOCLSALE17,-9999.0
430841,NaN,AK,Aleutian Islands Area,DIRSALES12,-9999.0
430842,NaN,AK,Aleutian Islands Area,DIRSALES17,-9999.0
430843,NaN,AK,Aleutian Islands Area,PCH_DIRSALES_12_17,-9999.0


In [6]:
print(missing_fips["State"].unique())
print("Unique counties/labels:", missing_fips["County"].nunique())

display(
    missing_fips[["State", "County", "Variable_Code"]]
    .drop_duplicates()
    .head(30)
)

['AK']
Unique counties/labels: 5


,State,County,Variable_Code
430834,AK,Aleutian Islands Area,DIRSALES_FARMS12
430835,AK,Aleutian Islands Area,DIRSALES_FARMS17
430836,AK,Aleutian Islands Area,PCH_DIRSALES_FARMS_12_17
430837,AK,Aleutian Islands Area,PCT_LOCLFARM12
430838,AK,Aleutian Islands Area,PCT_LOCLFARM17
430839,AK,Aleutian Islands Area,PCT_LOCLSALE12
430840,AK,Aleutian Islands Area,PCT_LOCLSALE17
430841,AK,Aleutian Islands Area,DIRSALES12
430842,AK,Aleutian Islands Area,DIRSALES17
430843,AK,Aleutian Islands Area,PCH_DIRSALES_12_17


In [7]:
missing_fips_summary = (
    missing_fips
    .groupby(["State", "County"])
    .agg(
        rows=("Variable_Code", "size"),
        unique_variables=("Variable_Code", "nunique")
    )
    .reset_index()
)

display(missing_fips_summary)

,State,County,rows,unique_variables
0,AK,Aleutian Islands Area,95,95
1,AK,Anchorage Area,95,95
2,AK,Fairbanks Area,95,95
3,AK,Juneau Area,95,95
4,AK,Kenai Peninsula Area,95,95


In [8]:
selected_predictors = [
    "PCT_LACCESS_POP19",
    "PCT_LACCESS_LOWI19",
    "GROCPTH20",
    "CONVSPTH20",
    "FFRPTH20",
    "FSRPTH20",
    "MEDHHINC21",
    "POVRATE21",
    "CHILDPOVRATE21",
    "DEEPPOVRATE21",
    "PC_SNAPBEN22",
    "PCT_65OLDER20",
    "PCT_18YOUNGER20",
    "PCT_NHWHITE20",
    "PCT_NHBLACK20",
    "PCT_HISP20",
    "PCT_NHASIAN20",
    "RECFACPTH20"
]

missing_fips_selected = missing_fips[
    missing_fips["Variable_Code"].isin(selected_predictors)
]

display(missing_fips_selected)
print(
    "Selected-predictor rows with missing FIPS:",
    len(missing_fips_selected)
)

,FIPS,State,County,Variable_Code,Value


Selected-predictor rows with missing FIPS: 0


In [9]:
available_selected = [
    predictor
    for predictor in selected_predictors
    if predictor in usda["Variable_Code"].unique()
]

missing_selected = [
    predictor
    for predictor in selected_predictors
    if predictor not in usda["Variable_Code"].unique()
]

print("Expected predictors:", len(selected_predictors))
print("Found predictors:", len(available_selected))
print("Missing predictors:", len(missing_selected))

print("\nFound:")
for predictor in available_selected:
    print("✓", predictor)

print("\nMissing:")
for predictor in missing_selected:
    print("✗", predictor)

Expected predictors: 18
Found predictors: 18
Missing predictors: 0

Found:
✓ PCT_LACCESS_POP19
✓ PCT_LACCESS_LOWI19
✓ GROCPTH20
✓ CONVSPTH20
✓ FFRPTH20
✓ FSRPTH20
✓ MEDHHINC21
✓ POVRATE21
✓ CHILDPOVRATE21
✓ DEEPPOVRATE21
✓ PC_SNAPBEN22
✓ PCT_65OLDER20
✓ PCT_18YOUNGER20
✓ PCT_NHWHITE20
✓ PCT_NHBLACK20
✓ PCT_HISP20
✓ PCT_NHASIAN20
✓ RECFACPTH20

Missing:


In [10]:
selected_usda = usda[
    usda["Variable_Code"].isin(selected_predictors)
].copy()

print("Rows for selected predictors:", len(selected_usda))
print("Unique FIPS:", selected_usda["FIPS"].nunique())

print("\nSpecial/missing values:")
print("NaN:", selected_usda["Value"].isna().sum())
print("-9999:", (selected_usda["Value"] == -9999).sum())
print("-8888:", (selected_usda["Value"] == -8888).sum())

Rows for selected predictors: 56685
Unique FIPS: 3156

Special/missing values:
NaN: 0
-9999: 3855
-8888: 111


In [11]:
special_by_predictor = (
    selected_usda
    .groupby("Variable_Code")
    .agg(
        rows=("Value", "size"),
        missing=("Value", lambda x: x.isna().sum()),
        minus_9999=("Value", lambda x: (x == -9999).sum()),
        minus_8888=("Value", lambda x: (x == -8888).sum())
    )
    .reindex(selected_predictors)
)

display(special_by_predictor)

,rows,missing,minus_9999,minus_8888
Variable_Code,,,,
PCT_LACCESS_POP19,3144,0,2,1
PCT_LACCESS_LOWI19,3144,0,2,1
GROCPTH20,3144,0,909,1
CONVSPTH20,3144,0,284,1
FFRPTH20,3144,0,471,1
FSRPTH20,3144,0,282,1
MEDHHINC21,3153,0,1,10
POVRATE21,3153,0,1,10
CHILDPOVRATE21,3153,0,1,10


In [12]:
coverage = (
    selected_usda
    .groupby("Variable_Code")
    .agg(
        rows=("FIPS", "size"),
        unique_fips=("FIPS", "nunique")
    )
    .reindex(selected_predictors)
)

display(coverage)

,rows,unique_fips
Variable_Code,,
PCT_LACCESS_POP19,3144,3144
PCT_LACCESS_LOWI19,3144,3144
GROCPTH20,3144,3144
CONVSPTH20,3144,3144
FFRPTH20,3144,3144
FSRPTH20,3144,3144
MEDHHINC21,3153,3153
POVRATE21,3153,3153
CHILDPOVRATE21,3153,3153


In [13]:
duplicates = (
    selected_usda
    .groupby(["Variable_Code", "FIPS"])
    .size()
    .reset_index(name="count")
)

duplicates = duplicates[duplicates["count"] > 1]

print("Duplicate predictor-FIPS combinations:", len(duplicates))
display(duplicates.head(20))

Duplicate predictor-FIPS combinations: 0


,Variable_Code,FIPS,count


In [14]:
minus_8888_rows = selected_usda[
    selected_usda["Value"] == -8888
]

print("Total -8888 cells:", len(minus_8888_rows))
print("Unique FIPS affected:", minus_8888_rows["FIPS"].nunique())

display(
    minus_8888_rows[
        ["FIPS", "State", "County", "Variable_Code", "Value"]
    ].sort_values(["FIPS", "Variable_Code"])
)

Total -8888 cells: 111
Unique FIPS affected: 13


,FIPS,State,County,Variable_Code,Value
767523,2261.0,AK,Valdez-Cordova,CHILDPOVRATE21,-8888.0
826159,2261.0,AK,Valdez-Cordova,CONVSPTH20,-8888.0
767521,2261.0,AK,Valdez-Cordova,DEEPPOVRATE21,-8888.0
729221,2261.0,AK,Valdez-Cordova,FFRPTH20,-8888.0
729227,2261.0,AK,Valdez-Cordova,FSRPTH20,-8888.0
...,...,...,...,...,...
771686,9190.0,CT,Western Connecticut,PCT_NHBLACK20,-8888.0
771685,9190.0,CT,Western Connecticut,PCT_NHWHITE20,-8888.0
771696,9190.0,CT,Western Connecticut,POVRATE21,-8888.0
335101,46113.0,SD,Shannon,PC_SNAPBEN22,-8888.0


In [15]:
print("Unique counties affected by -8888:", minus_8888_rows["FIPS"].nunique())

Unique counties affected by -8888: 13


In [16]:
missingness_inspection = special_by_predictor.copy()

missingness_inspection["missing_or_9999"] = (
    missingness_inspection["missing"]
    + missingness_inspection["minus_9999"]
)

missingness_inspection["missing_pct"] = (
    missingness_inspection["missing_or_9999"]
    / missingness_inspection["rows"]
    * 100
)

display(
    missingness_inspection[
        ["rows", "missing_or_9999", "missing_pct", "minus_8888"]
    ].round(2)
)

,rows,missing_or_9999,missing_pct,minus_8888
Variable_Code,,,,
PCT_LACCESS_POP19,3144,2,0.06,1
PCT_LACCESS_LOWI19,3144,2,0.06,1
GROCPTH20,3144,909,28.91,1
CONVSPTH20,3144,284,9.03,1
FFRPTH20,3144,471,14.98,1
FSRPTH20,3144,282,8.97,1
MEDHHINC21,3153,1,0.03,10
POVRATE21,3153,1,0.03,10
CHILDPOVRATE21,3153,1,0.03,10


In [17]:
variable_list_path = DATA_RAW / "VariableList.csv"

variable_list = pd.read_csv(variable_list_path)

print("Variable list shape:", variable_list.shape)
print("\nColumns:")
print(variable_list.columns.tolist())

display(variable_list.head())

Variable list shape: (304, 6)

Columns:
['Variable_Name', 'Category_Name', 'Category_Code', 'Subcategory_Name', 'Variable_Code', 'Units']


,Variable_Name,Category_Name,Category_Code,Subcategory_Name,Variable_Code,Units
0,"Population, low access to store, 2015",Access and Proximity to Foodstore,ACCESS,Overall,LACCESS_POP15,Count
1,"Population, low access to store, 2019",Access and Proximity to Foodstore,ACCESS,Overall,LACCESS_POP19,Count
2,"Population, low access to store (% change), 20...",Access and Proximity to Foodstore,ACCESS,Overall,PCH_LACCESS_POP_15_19,% change
3,"Population, low access to store (%), 2015",Access and Proximity to Foodstore,ACCESS,Overall,PCT_LACCESS_POP15,Percent
4,"Population, low access to store (%), 2019",Access and Proximity to Foodstore,ACCESS,Overall,PCT_LACCESS_POP19,Percent


In [18]:
selected_variable_info = (
    variable_list[
        variable_list["Variable_Code"].isin(selected_predictors)
    ][
        [
            "Variable_Code",
            "Variable_Name",
            "Category_Name",
            "Subcategory_Name",
            "Units"
        ]
    ]
    .copy()
)

# Keep them in the same order as our thesis predictor list
selected_variable_info["Variable_Code"] = pd.Categorical(
    selected_variable_info["Variable_Code"],
    categories=selected_predictors,
    ordered=True
)

selected_variable_info = selected_variable_info.sort_values("Variable_Code")

display(selected_variable_info)
print("Variables documented:", len(selected_variable_info))

,Variable_Code,Variable_Name,Category_Name,Subcategory_Name,Units
4,PCT_LACCESS_POP19,"Population, low access to store (%), 2019",Access and Proximity to Foodstore,Overall,Percent
9,PCT_LACCESS_LOWI19,"Low income & low access to store (%), 2019",Access and Proximity to Foodstore,Household Resources,Percent
69,GROCPTH20,"Grocery stores/1,000 pop, 2020",Store Availability,Grocery,"# per 1,000 pop"
81,CONVSPTH20,"Convenience stores/1,000 pop, 2020",Store Availability,Convenience,"# per 1,000 pop"
107,FFRPTH20,"Fast-food restaurants/1,000 pop, 2020",Restaurant Availability,Fast-food,"# per 1,000 pop"
113,FSRPTH20,"Full-service restaurants/1,000 pop, 2020",Restaurant Availability,Full-service,Count
296,MEDHHINC21,"Median household income, 2021",Socioeconomic Characteristics,Income Level,Dollars
297,POVRATE21,"Poverty rate, 2021",Socioeconomic Characteristics,Income Level,Percent
300,CHILDPOVRATE21,"Child poverty rate, 2021",Socioeconomic Characteristics,Income Level,Percent
298,DEEPPOVRATE21,"Deep poverty rate, 2021",Socioeconomic Characteristics,Income Level,Percent


Variables documented: 18


In [19]:
cdc_files = list(DATA_RAW.glob("PLACES*.csv"))

print("CDC files found:", len(cdc_files))

for file in cdc_files:
    print(file.name)

CDC files found: 1
PLACES__County_Data_(GIS_Friendly_Format),_2024_release_20260824.csv


In [20]:
cdc_path = cdc_files[0]

cdc = pd.read_csv(cdc_path)

print("CDC shape:", cdc.shape)
display(cdc.head())

CDC shape: (3144, 167)


,StateAbbr,StateDesc,CountyName,CountyFIPS,TotalPopulation,TotalPop18plus,ACCESS2_CrudePrev,ACCESS2_Crude95CI,ACCESS2_AdjPrev,ACCESS2_Adj95CI,...,SHUTUTILITY_Adj95CI,LACKTRPT_CrudePrev,LACKTRPT_Crude95CI,LACKTRPT_AdjPrev,LACKTRPT_Adj95CI,EMOTIONSPT_CrudePrev,EMOTIONSPT_Crude95CI,EMOTIONSPT_AdjPrev,EMOTIONSPT_Adj95CI,Geolocation
0,IL,Illinois,Hamilton,17065,"7,984","6,181",7.1,"( 6.3, 8.1)",9.7,"( 8.6, 11.1)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-88.5390680927951 38.0815781203156)
1,NY,New York,Seneca,36099,"32,882","26,182",5.2,"( 4.7, 5.8)",7.4,"( 6.8, 8.2)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-76.8234304711163 42.7807211049499)
2,OK,Oklahoma,Haskell,40061,"11,641","8,933",10.7,"( 9.5, 11.9)",14.1,"(12.4, 15.8)",...,"(11.1, 14.4)",11.1,"(10.0, 12.2)",12.0,"(10.8, 13.1)",25.3,"(21.5, 29.3)",26.0,"(22.2, 30.1)",POINT (-95.116494065308 35.224815744206)
3,IL,Illinois,Lake,17097,"709,150","546,991",10.1,"( 9.0, 11.3)",10.4,"( 9.2, 11.7)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-88.003802278105 42.3231861408737)
4,KS,Kansas,Atchison,20005,"16,108","12,602",7.2,"( 6.3, 8.2)",9.5,"( 8.4, 10.7)",...,"( 7.9, 10.2)",7.9,"( 7.2, 8.8)",8.3,"( 7.5, 9.2)",21.4,"(18.4, 24.7)",21.8,"(18.8, 25.2)",POINT (-95.3139519927859 39.5314729069169)


In [21]:
print("CDC shape:", cdc.shape)

print("\nTarget exists:")
print("OBESITY_AdjPrev" in cdc.columns)

print("\nCountyFIPS dtype:")
print(cdc["CountyFIPS"].dtype)

print("\nOBESITY_AdjPrev dtype:")
print(cdc["OBESITY_AdjPrev"].dtype)

CDC shape: (3144, 167)

Target exists:
True

CountyFIPS dtype:
int64

OBESITY_AdjPrev dtype:
float64


In [22]:

print("Total rows:", len(cdc))
print("Unique CountyFIPS:", cdc["CountyFIPS"].nunique())
print("Duplicate CountyFIPS:", cdc["CountyFIPS"].duplicated().sum())
print("Missing CountyFIPS:", cdc["CountyFIPS"].isna().sum())
print("Missing OBESITY_AdjPrev:", cdc["OBESITY_AdjPrev"].isna().sum())

Total rows: 3144
Unique CountyFIPS: 3144
Duplicate CountyFIPS: 0
Missing CountyFIPS: 0
Missing OBESITY_AdjPrev: 0


In [23]:
display(
    cdc[
        [
            "StateAbbr",
            "StateDesc",
            "CountyName",
            "CountyFIPS",
            "OBESITY_AdjPrev"
        ]
    ].head(10)
)

,StateAbbr,StateDesc,CountyName,CountyFIPS,OBESITY_AdjPrev
0,IL,Illinois,Hamilton,17065,37.8
1,NY,New York,Seneca,36099,37.4
2,OK,Oklahoma,Haskell,40061,43.7
3,IL,Illinois,Lake,17097,31.9
4,KS,Kansas,Atchison,20005,40.5
5,AK,Alaska,Anchorage,2020,29.6
6,OK,Oklahoma,Seminole,40133,43.0
7,WV,West Virginia,Mercer,54055,42.5
8,MN,Minnesota,Blue Earth,27013,42.1
9,NE,Nebraska,Buffalo,31019,43.4


In [24]:
print(cdc["OBESITY_AdjPrev"].describe())

count    3144.000000
mean       37.907697
std         4.647178
min        17.700000
25%        35.300000
50%        38.400000
75%        41.000000
max        53.000000
Name: OBESITY_AdjPrev, dtype: float64


In [25]:
invalid_obesity = cdc[
    (cdc["OBESITY_AdjPrev"] < 0) |
    (cdc["OBESITY_AdjPrev"] > 100)
]

print("Invalid obesity prevalence values:", len(invalid_obesity))

Invalid obesity prevalence values: 0
